# 16 · Graph RAG

**Idea:** instead of retrieving by *similarity*, retrieve by *connection*. An LLM extracts `subject | relation | object` triples from each chunk; we build a directed **entity graph** (networkx). At query time we find the entities named in the question, expand to their neighborhood, and return the chunks attached to those nodes — surfacing linked facts a vector search would scatter.

> Extraction runs one LLM call per chunk, so it defaults to Ollama.

In [1]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [2]:
from rag_lab import load_markdown, GraphRAG

# Use the systems corpus — it is full of named entities and relations.
systems = [c for c in load_markdown() if c.source == 'systems_and_teams.md']
graph = GraphRAG(provider='ollama')
graph.add(systems)
print('nodes:', len(graph), '| edges:', graph.graph.number_of_edges())

---------------------------------
Atlas is the retrieval service that powers the Helpdesk Assistant. It indexes the
support knowledge base, embeds passages with the nomic-embed-text model, and stores
them in a Chroma vector database. Atlas was built by the Search Platform team and
calls the Beacon reranker before returning results.
{'heading': 'Atlas retrieval service', 'title': 'RAG Systems Field Notes', 'tags': ['systems', 'case-study', 'architecture'], 'audience': 'engineers'}
---------------------------------
Processing chunk 0: Atlas is the retrieval service that powers the Helpdesk Assistant. It indexes the
support knowledge base, embeds passages with the nomic-embed-text model, and stores
them in a Chroma vector database. Atlas was built by the Search Platform team and
calls the Beacon reranker before returning results.
Atlas | is | retrieval service
########################################################################
Adding node: atlas -> Atlas
{'atlas': {0}}
##############

Inspect the extracted relationships near an entity — this is the knowledge the LLM pulled out of the prose.

In [3]:
for s, r, o in graph.edges_near('Atlas', depth=1):
    print(f'{s}  --[{r}]-->  {o}')

DEBUG: Finding neighborhood for query 'Atlas' with depth 1
DEBUG: Found initial nodes: {'atlas'}
DEBUG: Initial frontier: {'atlas'}
DEBUG: Expanding at depth 0
DEBUG: Node atlas has successors: {'support knowledge base', 'results', 'beacon', 'retrieval service', 'helpdesk assistant', 'passages', 'nomic-embed-text model', 'beacon reranker', 'search platform team'}, predecessors: {'the search platform team'}
Atlas  --[is]-->  retrieval service
Atlas  --[powers]-->  Helpdesk Assistant
Atlas  --[indexes]-->  support knowledge base
Atlas  --[stores]-->  passages
Atlas  --[uses]-->  nomic-embed-text model
Atlas  --[built by]-->  Search Platform team
Atlas  --[calls]-->  Beacon reranker
Atlas  --[returns]-->  results
Atlas  --[calls]-->  Beacon
Search Platform team  --[built]-->  Beacon
Beacon  --[is]-->  reranking component
Beacon  --[reorders]-->  top candidates
Beacon  --[is trained on]-->  labeled support tickets
Beacon  --[was built by]-->  Search Platform team
The Search Platform team  

Now retrieve by traversal: name an entity in the query and the graph returns the chunks about it **and its neighbors** (`depth=1`).

In [4]:
for h in graph.search('What is Beacon and who built it?', k=3, depth=1):
    print(f'{h.score:.0f}  {h.chunk.meta.get("heading")}')
    print('   ', h.chunk.text[:120], '...')
    print()

DEBUG: Finding neighborhood for query 'What is Beacon and who built it?' with depth 1
DEBUG: Found initial nodes: {'beacon'}
DEBUG: Initial frontier: {'beacon'}
DEBUG: Expanding at depth 0
DEBUG: Node beacon has successors: {'search platform team', 'top candidates', 'reranking component', 'labeled support tickets'}, predecessors: {'the search platform team', 'search platform team', 'atlas'}
6  Beacon reranker
    Beacon is the reranking component that Atlas calls to reorder its top candidates
before they reach the generator. Beacon ...

3  Search Platform team
    The Search Platform team owns both Atlas and Beacon. The team is led by Priya Nair
and focuses on retrieval quality, ind ...

2  Atlas retrieval service
    Atlas is the retrieval service that powers the Helpdesk Assistant. It indexes the
support knowledge base, embeds passage ...



**Takeaway:** graph RAG makes *relationships* first-class. It excels at multi-entity and "how are these connected?" questions where pure similarity search returns disconnected fragments. The cost is an extraction pass over the corpus — done once at index time.

Next → `17_citation_generation.ipynb`